In [0]:
!pip install torch torchvision opencv-python transformers

In [0]:

# 1. IMPORT LIBRARIES & SET INLINE PLOTTING

%matplotlib inline
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import io
import torch
import requests

from transformers import Owlv2Processor, Owlv2ForObjectDetection


# 2. DEFINE PATHS TO IMAGE FRAME

image_path = "/xxx/Eem/Decoded Frames/Eem_ch04_0619_060343_235956/video1/0000001.jpg" # Path to the initial image you want to apply the object detection and segmentation


# 3. INITIALIZE THE OWLv2 MODEL

processor = Owlv2Processor.from_pretrained("google/owlv2-base-patch16-ensemble")
model = Owlv2ForObjectDetection.from_pretrained("google/owlv2-base-patch16-ensemble")


# 4. LOAD THE IMAGE AND RUN PREDICTION USING OWLv2

# Load image using PIL
image = Image.open(image_path)

# Define text queries; here we use a set of common object prompts.
# You can adjust or extend this list as needed.
texts = [["Cow"]]

# Prepare inputs for the model
inputs = processor(text=texts, images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# Rescale predicted bounding boxes to the image size
target_sizes = torch.Tensor([image.size[::-1]])
results = processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes, threshold=0.35)


# 5. PRINT OUT THE PREDICTION DETAILS

print("Detected Objects:")
for box, score, label in zip(results[0]["boxes"], results[0]["scores"], results[0]["labels"]):
    coords = [round(x, 2) for x in box.tolist()]  # [xmin, ymin, xmax, ymax]
    # Get the class name from the provided text queries based on the label index
    class_name = texts[0][label] if label < len(texts[0]) else str(label)
    print(f"Class: {class_name}, Confidence: {round(score.item(), 2)}, BBox: {coords}")


# 6. VISUALIZE THE BOUNDING BOXES ON THE ORIGINAL IMAGE

# Load the original image using OpenCV (BGR format)
image_cv2 = cv2.imread(image_path)
if image_cv2 is None:
    raise ValueError(f"Could not load the image from {image_path}")

# Loop through each detected bounding box and draw on the image
for box, score, label in zip(results[0]["boxes"], results[0]["scores"], results[0]["labels"]):
    x1, y1, x2, y2 = map(int, box.tolist())
    conf = score.item()
    class_name = texts[0][label] if label < len(texts[0]) else str(label)
    label_text = f"{class_name}: {conf:.2f}"
    
    # Draw the bounding box (green rectangle)
    cv2.rectangle(image_cv2, (x1, y1), (x2, y2), color=(0, 255, 0), thickness=2)
    # Draw a label background for better visibility
    (text_width, text_height), baseline = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(image_cv2, (x1, y1 - text_height - baseline), (x1 + text_width, y1), (0, 255, 0), -1)
    # Put the label text above the bounding box
    cv2.putText(image_cv2, label_text, (x1, y1 - baseline), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), thickness=1)

# Convert image from BGR to RGB for proper display with matplotlib and PIL
image_rgb = cv2.cvtColor(image_cv2, cv2.COLOR_BGR2RGB)


# 7A. DISPLAY THE PLOT USING MATPLOTLIB

plt.figure(figsize=(10, 10))
plt.imshow(image_rgb)
plt.axis("off")
plt.title("Detection Results on Original Image")
plt.show()


# 7B. IF THE MATPLOTLIB PLOT DOESN'T APPEAR, USE THE DATBRICKS display() FUNCTION

# Convert the NumPy image array to a PIL image and display it:
pil_img = Image.fromarray(image_rgb)
display(pil_img)